In [ ]:
import os, platform, sys
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

require(os.path.exists('/content'), 'Open this notebook in Google Colab.')
require((3, 10) <= sys.version_info[:2] < (3, 14), 'Use Colab Python 3.10 through 3.13.')
print('Python', platform.python_version(), '| Colab environment ready')


In [ ]:
!rm -rf /content/babel
!git clone --quiet https://github.com/dhelmy990/babel.git /content/babel
!git -C /content/babel checkout --quiet 92f3ac697d78eb827d75b033df92dcbed887def7
!python -m pip install --quiet --require-hashes -r /content/babel/training/requirements-colab.lock
!python -m pip install --quiet --no-deps -e /content/babel/training
REPOSITORY_URL = 'https://github.com/dhelmy990/babel.git'
SOURCE_COMMIT_SHA = '92f3ac697d78eb827d75b033df92dcbed887def7'


In [ ]:
import subprocess
installed_source_sha = subprocess.check_output(
    ['git', '-C', '/content/babel', 'rev-parse', 'HEAD'], text=True
).strip()
require(installed_source_sha == SOURCE_COMMIT_SHA, 'Installed Babel source commit mismatch')
from babel_training.config import DistillationConfig
print('Pinned Babel source imported:', installed_source_sha)


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN under the key icon in the Colab left sidebar.')
print('Private Hub credential loaded from Colab Secrets (value hidden).')


In [ ]:
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
run_id = datetime.now(timezone.utc).strftime('interview-50k-%Y%m%dT%H%M%SZ')
drive_root = '/content/drive/MyDrive/babel-distillation/interview-50k'
run_root = os.path.join(drive_root, run_id)
os.makedirs(run_root, exist_ok=False)
print('Drive run directory:', run_root)


In [ ]:
DATASET_REPO_ID = 'dhelmy990/babel-wikipedia-experiment'
DATASET_CONFIG = 'distillation_2016_interview'
DATASET_REVISION = 'b440e98b04ab77afed7caf0455eca3189235fc3b'
MODEL_REVISION = '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3'
MANIFEST_SHA256 = '33c65554da38af5888e5aae75350ae8ee7889d6047c9f8339d97781e4326de09'
TRAIN_ORDERED_SHA256 = '518c30f10859a88681c3708ab0236bd104fdde96acff09089515d871d9600a1e'
VALIDATION_ORDERED_SHA256 = '64cd7c82c58d73947f24b8120ef3c2e5c3a4a8f145bf0a7a6522175bcd1b2cd6'
TEST_ORDERED_SHA256 = 'd2cd61ee895c2f6386c708d7884666b4aa579174674e8bf70e876ee891956bf5'
TRAIN_PARQUET_SHA256 = '11a217879913305a88b0bfaafffa39f132883d2b6f27252a02054ba95ea6b2c5'
VALIDATION_PARQUET_SHA256 = 'a925eb795f253635f3a80a76994a7139a0f81f4e784beb61dfc93f8b662dc8f0'
TEST_PARQUET_SHA256 = '103f22b38b048973f8ab6ba52efca41f667f37c305b5dfea8b752dc492d7ac03'
EXPECTED_COUNTS = {'total': 60_000, 'train': 50_000, 'validation': 5_000, 'test': 5_000}
EXPECTED_ORDERED_SHA256 = {
    'train': TRAIN_ORDERED_SHA256,
    'validation': VALIDATION_ORDERED_SHA256,
    'test': TEST_ORDERED_SHA256,
}
EXPECTED_PARQUET_SHA256 = {
    'train': TRAIN_PARQUET_SHA256,
    'validation': VALIDATION_PARQUET_SHA256,
    'test': TEST_PARQUET_SHA256,
}
SMOKE_ROWS = 1_000
TRAIN_ROWS = 50_000
VALIDATION_ROWS = 5_000
TEST_COUNT = 5_000
EPOCHS = 1


In [ ]:
config = DistillationConfig(max_length=384)
if config.model_revision != MODEL_REVISION:
    raise RuntimeError('Pinned Qwen revision mismatch')
per_device_batch_size = 2
gradient_accumulation_steps = 8
smoke_gradient_accumulation_steps = 4
checkpoint_interval = 100
max_runtime_minutes = None
TRAINING_SEED = 7
DESTINATION_MODEL_REPO = 'dhelmy990/babel-qwen-navigation-2016-interview'
RESUME_CHECKPOINT_DIR = None
training_config = {
    'config_version': 1,
    'source_commit_sha': SOURCE_COMMIT_SHA,
    'model_id': config.model_id,
    'model_revision': config.model_revision,
    'tokenizer_revision': config.model_revision,
    'dataset_repo_id': DATASET_REPO_ID,
    'dataset_config': DATASET_CONFIG,
    'dataset_commit_sha': DATASET_REVISION,
    'dataset_manifest_sha256': MANIFEST_SHA256,
    'ordered_identity_sha256': EXPECTED_ORDERED_SHA256,
    'parquet_sha256': EXPECTED_PARQUET_SHA256,
    'teacher_dimension': config.teacher_dimension,
    'projection_input_dimension': 1024,
    'projection_output_dimension': 100,
    'max_length': config.max_length,
    'lambda_rel': config.lambda_rel,
    'lora_rank': config.lora_rank,
    'lora_alpha': config.lora_alpha,
    'lora_dropout': config.lora_dropout,
    'lora_targets': list(config.lora_targets),
    'lora_bias': 'none',
    'per_device_batch_size': per_device_batch_size,
    'gradient_accumulation_steps': gradient_accumulation_steps,
    'smoke_gradient_accumulation_steps': smoke_gradient_accumulation_steps,
    'checkpoint_interval_optimizer_steps': checkpoint_interval,
    'smoke_rows': SMOKE_ROWS,
    'train_rows': TRAIN_ROWS,
    'validation_rows': VALIDATION_ROWS,
    'epochs': EPOCHS,
    'seed': TRAINING_SEED,
}


In [ ]:
import hashlib, json
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
def require_identity(condition, message):
    if not condition:
        raise RuntimeError(message)

hub_api = HfApi()
dataset_metadata = hub_api.dataset_info(
    DATASET_REPO_ID, revision=DATASET_REVISION, token=HF_TOKEN
)
dataset_revision = dataset_metadata.sha
require_identity(dataset_revision == DATASET_REVISION, 'Pinned dataset revision mismatch')
require_identity(getattr(dataset_metadata, 'private', False) is True, 'Dataset repository is not private')
manifest_path = Path(hf_hub_download(
    DATASET_REPO_ID,
    filename=f'{DATASET_CONFIG}/manifest.json',
    repo_type='dataset',
    revision=dataset_revision,
    token=HF_TOKEN,
))
manifest_bytes = manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
require_identity(manifest_sha256 == MANIFEST_SHA256, 'Manifest SHA-256 mismatch')
manifest = json.loads(manifest_bytes)
require_identity(manifest['dataset_config'] == DATASET_CONFIG, 'Dataset configuration mismatch')
require_identity(manifest['state'] == 'interview_ready', 'Dataset is not interview-ready')
require_identity(manifest['schema'] == 'distillation-example-v1', 'Dataset schema mismatch')
require_identity(manifest['counts'] == EXPECTED_COUNTS, 'Dataset counts mismatch')
require_identity(manifest['selection']['seed'] == 'babel-interview-2016-v1', 'Selection seed mismatch')
require_identity(manifest['selection']['ordered_identity_sha256'] == EXPECTED_ORDERED_SHA256, 'Ordered identity checksum mismatch')
require_identity(len(manifest['selection']['smoke_article_keys']) == SMOKE_ROWS, 'Smoke prefix count mismatch')
require_identity(manifest['frontier']['complete_corpus'] is False, 'Frontier disclosure mismatch')
shard_sha256 = {item['split']: item['sha256'] for item in manifest['shards']}
require_identity(shard_sha256 == EXPECTED_PARQUET_SHA256, 'Parquet checksum declaration mismatch')
remote_tree = {
    entry.path: entry
    for entry in hub_api.list_repo_tree(
        DATASET_REPO_ID,
        path_in_repo=DATASET_CONFIG,
        recursive=True,
        expand=True,
        revision=dataset_revision,
        repo_type='dataset',
        token=HF_TOKEN,
    )
    if hasattr(entry, 'path')
}
for shard in manifest['shards']:
    remote_file = remote_tree[shard['path']]
    require_identity(remote_file.lfs is not None, f"Shard is not LFS-backed: {shard['path']}")
    require_identity(remote_file.lfs.sha256 == shard['sha256'], f"Remote shard SHA-256 mismatch: {shard['path']}")
    require_identity(remote_file.lfs.size == shard['bytes'], f"Remote shard size mismatch: {shard['path']}")
require_identity(manifest['selection']['ordered_identity_sha256']['train'] == TRAIN_ORDERED_SHA256, 'Train ordered identity mismatch')
require_identity(manifest['selection']['ordered_identity_sha256']['validation'] == VALIDATION_ORDERED_SHA256, 'Validation ordered identity mismatch')
require_identity(manifest['selection']['ordered_identity_sha256']['test'] == TEST_ORDERED_SHA256, 'Test metadata identity mismatch')
readiness_path = Path(hf_hub_download(
    DATASET_REPO_ID,
    filename=f'{DATASET_CONFIG}/readiness.json',
    repo_type='dataset',
    revision=dataset_revision,
    token=HF_TOKEN,
))
readiness_bytes = readiness_path.read_bytes()
dataset_readiness_sha256 = hashlib.sha256(readiness_bytes).hexdigest()
readiness = json.loads(readiness_bytes)
require_identity(readiness['manifest_sha256'] == MANIFEST_SHA256, 'Readiness manifest mismatch')
require_identity(readiness['counts'] == EXPECTED_COUNTS, 'Readiness counts mismatch')
print('Immutable dataset identity gates PASS; test identity checked as metadata only.')


In [ ]:
from itertools import islice
from datasets import load_dataset
from babel_training.data import validate_distillation_row

class OrderedInterviewStream:
    def __init__(self, split, row_limit):
        if split not in {'train', 'validation'}:
            raise ValueError('ordered stream split must be train or validation')
        if not isinstance(row_limit, int) or isinstance(row_limit, bool) or row_limit <= 0:
            raise ValueError('ordered stream row_limit must be a positive integer')
        self.split = split
        self.row_limit = row_limit
        self.next_ordered_row = 0

    def state_dict(self):
        return {
            'state_version': 1,
            'dataset_repo_id': DATASET_REPO_ID,
            'dataset_config': DATASET_CONFIG,
            'dataset_revision': dataset_revision,
            'split': self.split,
            'row_limit': self.row_limit,
            'next_ordered_row': self.next_ordered_row,
        }

    def load_state_dict(self, state):
        expected = {
            'state_version': 1,
            'dataset_repo_id': DATASET_REPO_ID,
            'dataset_config': DATASET_CONFIG,
            'dataset_revision': dataset_revision,
            'split': self.split,
            'row_limit': self.row_limit,
        }
        require({name: state[name] for name in expected} == expected, 'Stream checkpoint identity mismatch')
        next_ordered_row = state['next_ordered_row']
        require(isinstance(next_ordered_row, int), 'Stream checkpoint cursor must be an integer')
        require(0 <= next_ordered_row <= self.row_limit, 'Stream checkpoint cursor is out of range')
        self.next_ordered_row = next_ordered_row

    def __iter__(self):
        start = self.next_ordered_row
        ordered_dataset = load_dataset(
            DATASET_REPO_ID,
            DATASET_CONFIG,
            split=self.split,
            revision=dataset_revision,
            token=HF_TOKEN,
            streaming=True,
        )
        remaining = self.row_limit - start
        for offset, raw in enumerate(islice(ordered_dataset.skip(start), remaining), start=1):
            validated = validate_distillation_row(raw, expected_split=self.split)
            checked = {
                name: validated[name]
                for name in (
                    'article_key', 'page_id', 'canonical_title', 'lead_text',
                    'teacher_vector', 'teacher_norm', 'split',
                )
            }
            self.next_ordered_row = start + offset
            yield checked
        require(
            self.next_ordered_row == self.row_limit,
            f'{self.split} ended at {self.next_ordered_row}, expected {self.row_limit}',
        )

print('Ordered restartable train/validation stream factory ready; no test stream exists.')


In [ ]:
train_preview = list(islice(OrderedInterviewStream('train', TRAIN_ROWS), 2))
validation_preview = list(islice(
    OrderedInterviewStream('validation', VALIDATION_ROWS), 2
))
require([row['article_key'] for row in train_preview] == [
    item['article_key']
    for item in manifest['selection']['ordered_identities']['train'][:2]
], 'Train preview identity mismatch')
require([row['article_key'] for row in validation_preview] == [
    item['article_key']
    for item in manifest['selection']['ordered_identities']['validation'][:2]
], 'Validation preview identity mismatch')
print({
    'train': [
        {'article_key': row['article_key'], 'title': row['canonical_title']}
        for row in train_preview
    ],
    'validation': [
        {'article_key': row['article_key'], 'title': row['canonical_title']}
        for row in validation_preview
    ],
})


In [ ]:
import torch
require(torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.')
major, minor = torch.cuda.get_device_capability()
mixed_precision = 'bf16' if major >= 8 else 'fp16'
if major < 8:
    mixed_precision = 'fp16'
print(torch.cuda.get_device_name(0), '| mixed precision:', mixed_precision)


In [ ]:
import math
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from babel_training.collator import DistillationCollator
from babel_training.model import DistilledQwenEncoder
from babel_training.trainer import DistillationTrainer, build_stateful_train_loader

tokenizer = AutoTokenizer.from_pretrained(
    config.model_id, revision=config.model_revision, token=HF_TOKEN
)
collator = DistillationCollator(tokenizer, max_length=config.max_length)
gate_rows = list(islice(OrderedInterviewStream('validation', VALIDATION_ROWS), 2))
gate_loader = DataLoader(
    gate_rows,
    batch_size=per_device_batch_size,
    collate_fn=collator,
    num_workers=0,
)
validation_batch = next(iter(gate_loader))
smoke_train_stream = OrderedInterviewStream('train', SMOKE_ROWS)
smoke_train_loader = build_stateful_train_loader(
    smoke_train_stream,
    batch_size=per_device_batch_size,
    collate_fn=collator,
)
smoke_model = DistilledQwenEncoder.from_pretrained(config)
smoke_optimizer = AdamW(
    [parameter for parameter in smoke_model.parameters() if parameter.requires_grad],
    lr=2e-4,
)
smoke_scheduler = torch.optim.lr_scheduler.LambdaLR(
    smoke_optimizer, lr_lambda=lambda _: 1.0
)
smoke_trainer = DistillationTrainer(
    smoke_model,
    smoke_train_loader,
    validation_batch=validation_batch,
    model_id=config.model_id,
    model_revision=config.model_revision,
    dataset_revision=dataset_revision,
    training_config=training_config,
    optimizer=smoke_optimizer,
    scheduler=smoke_scheduler,
    mixed_precision=mixed_precision,
    gradient_accumulation_steps=smoke_gradient_accumulation_steps,
    max_runtime_minutes=max_runtime_minutes,
)
smoke_scaler = smoke_trainer.accelerator.scaler


In [ ]:
gate = smoke_trainer.one_batch_gate()
require(all(math.isfinite(value) for value in gate.values()), 'One-batch gate is non-finite')
require(gate['gradient_norm'] > 0, 'One-batch gate gradient norm is not positive')
print('ONE-BATCH NUMERICAL/GRADIENT GATE PASS', gate)


In [ ]:
smoke_ordered_prefix = list(islice(
    manifest['selection']['ordered_identities']['train'], SMOKE_ROWS
))
require([item['article_key'] for item in smoke_ordered_prefix] == (
    manifest['selection']['smoke_article_keys']
), 'Smoke prefix identity mismatch')
require(SMOKE_ROWS % per_device_batch_size == 0, 'Smoke rows must form complete microbatches')
smoke_microbatches = SMOKE_ROWS // per_device_batch_size
require(smoke_microbatches % smoke_gradient_accumulation_steps == 0, 'Smoke microbatches must form complete optimizer steps')
smoke_optimizer_steps = (
    smoke_microbatches // smoke_gradient_accumulation_steps
)
smoke_losses = smoke_trainer.train(max_steps=smoke_optimizer_steps)
require(smoke_trainer.global_step == smoke_optimizer_steps, 'Smoke optimizer step count mismatch')
require(smoke_train_stream.next_ordered_row == SMOKE_ROWS, 'Smoke row count mismatch')
smoke_trainer.epoch = 1
smoke_checkpoint_dir = os.path.join(run_root, 'smoke-checkpoint')
smoke_manifest = smoke_trainer.save(
    smoke_checkpoint_dir,
    metrics={'mode': 'smoke', 'rows': SMOKE_ROWS, 'final_loss': smoke_losses[-1]},
)
torch.save(
    {
        'optimizer': smoke_trainer.optimizer.state_dict(),
        'scheduler': smoke_trainer.scheduler.state_dict(),
        'scaler': smoke_scaler.state_dict() if smoke_scaler is not None else None,
        'rng': {
            'python': __import__('random').getstate(),
            'numpy': __import__('numpy').random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all(),
        },
        'epoch': smoke_trainer.epoch,
        'global_step': smoke_trainer.global_step,
        'next_ordered_row': smoke_train_stream.next_ordered_row,
    },
    os.path.join(smoke_checkpoint_dir, 'notebook_restart_state.pt'),
)
print('SMOKE CHECKPOINT saved after first ordered 1,000 train rows:', smoke_checkpoint_dir)


In [ ]:
import gc, random
import numpy as np

del smoke_trainer, smoke_model, smoke_optimizer, smoke_scheduler, smoke_scaler
del smoke_train_loader, smoke_train_stream
gc.collect()
torch.cuda.empty_cache()
random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)
production_train_stream = OrderedInterviewStream('train', TRAIN_ROWS)
require(production_train_stream.next_ordered_row == 0, 'Production loader cursor did not reset')
production_train_loader = build_stateful_train_loader(
    production_train_stream,
    batch_size=per_device_batch_size,
    collate_fn=collator,
)
production_model = DistilledQwenEncoder.from_pretrained(config)
production_optimizer = AdamW(
    [
        parameter
        for parameter in production_model.parameters()
        if parameter.requires_grad
    ],
    lr=2e-4,
)
production_scheduler = torch.optim.lr_scheduler.LambdaLR(
    production_optimizer, lr_lambda=lambda _: 1.0
)
production_trainer = DistillationTrainer(
    production_model,
    production_train_loader,
    validation_batch=validation_batch,
    model_id=config.model_id,
    model_revision=config.model_revision,
    dataset_revision=dataset_revision,
    training_config=training_config,
    optimizer=production_optimizer,
    scheduler=production_scheduler,
    mixed_precision=mixed_precision,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_runtime_minutes=max_runtime_minutes,
)
production_scaler = production_trainer.accelerator.scaler
require(production_trainer.global_step == 0, 'Production global step did not reset')
require(production_trainer.epoch == 0, 'Production epoch did not reset')
if RESUME_CHECKPOINT_DIR is not None:
    resume_checkpoint_path = Path(RESUME_CHECKPOINT_DIR).resolve(strict=True)
    resolved_drive_root = Path(drive_root).resolve(strict=True)
    if os.path.commonpath([resume_checkpoint_path, resolved_drive_root]) != str(resolved_drive_root):
        raise RuntimeError('Resume checkpoint must be beneath the configured Drive root')
    if not Path(resume_checkpoint_path, 'NOTEBOOK_CHECKPOINT_COMPLETE').is_file():
        raise RuntimeError('Resume checkpoint is incomplete')
    restored_checkpoint_manifest = json.loads(
        Path(resume_checkpoint_path, 'manifest.json').read_text(encoding='utf-8')
    )
    if restored_checkpoint_manifest['training_config'] != training_config:
        raise RuntimeError('Resume checkpoint training identity mismatch')
    production_trainer.reload(str(resume_checkpoint_path))
    restored_restart_state = json.loads(
        Path(resume_checkpoint_path, 'notebook_restart_metadata.json').read_text(encoding='utf-8')
    )
    if production_trainer.global_step != restored_restart_state['global_step']:
        raise RuntimeError('Restored global step mismatch')
    if production_trainer.epoch != restored_restart_state['epoch']:
        raise RuntimeError('Restored epoch mismatch')
    if production_train_stream.next_ordered_row != restored_restart_state['next_ordered_row']:
        raise RuntimeError('Restored ordered row cursor mismatch')
    if production_train_stream.next_ordered_row >= TRAIN_ROWS:
        raise RuntimeError('Resume checkpoint has no remaining production rows')
    print('Restored interrupted production run at ordered row', production_train_stream.next_ordered_row)
print('Production model, optimizer, scheduler, scaler, loader cursor, and RNG rebuilt.')


In [ ]:
QUICK_TEST_MODE = True
require(isinstance(QUICK_TEST_MODE, bool), 'QUICK_TEST_MODE must be an explicit boolean')
if QUICK_TEST_MODE is False:
    print('PRODUCTION OPT-IN CONFIRMED: exactly one ordered 50,000-row epoch will run.')
else:
    print('QUICK TEST MODE: exactly one optimizer step will run, save, and stop.')

def capture_restart_state(trainer, stream, scaler):
    return {
        'optimizer': trainer.optimizer.state_dict(),
        'scheduler': trainer.scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'rng': {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all(),
        },
        'epoch': trainer.epoch,
        'global_step': trainer.global_step,
        'next_ordered_row': stream.next_ordered_row,
    }

def complete_restartable_checkpoint(path, trainer, stream, scaler, manifest_value):
    restart_state = capture_restart_state(trainer, stream, scaler)
    require(restart_state['next_ordered_row'] == stream.state_dict()['next_ordered_row'], 'Checkpoint loader cursor mismatch')
    torch.save(restart_state, os.path.join(path, 'notebook_restart_state.pt'))
    restart_metadata = {
        'state_version': 1,
        'components': ['optimizer', 'scheduler', 'scaler', 'rng'],
        'epoch': restart_state['epoch'],
        'global_step': restart_state['global_step'],
        'next_ordered_row': restart_state['next_ordered_row'],
    }
    Path(path, 'notebook_restart_metadata.json').write_text(
        json.dumps(restart_metadata, sort_keys=True, separators=(',', ':')) + '\n',
        encoding='utf-8',
    )
    Path(path, 'NOTEBOOK_CHECKPOINT_COMPLETE').write_text(
        'optimizer scheduler scaler rng epoch global_step next_ordered_row\n',
        encoding='utf-8',
    )
    return manifest_value


In [ ]:
for training_pass in range(1):
    if QUICK_TEST_MODE:
        optimizer_step_limit = 1
        quick_start_step = production_trainer.global_step
        quick_losses = production_trainer.train(
            max_steps=quick_start_step + optimizer_step_limit
        )
        require(production_trainer.global_step == quick_start_step + 1, 'Quick test did not take exactly one optimizer step')
        quick_test_checkpoint_dir = os.path.join(run_root, 'quick-test')
        quick_metrics = {
            'mode': 'quick_test',
            'optimizer_steps': optimizer_step_limit,
            'next_ordered_row': production_train_stream.next_ordered_row,
            'loss': quick_losses[-1],
        }
        quick_manifest = production_trainer.save(
            quick_test_checkpoint_dir,
            metrics=quick_metrics,
        )
        quick_manifest = complete_restartable_checkpoint(
            quick_test_checkpoint_dir, production_trainer,
            production_train_stream, production_scaler, quick_manifest,
        )
        quick_restart_metadata = json.loads(
            Path(quick_test_checkpoint_dir, 'notebook_restart_metadata.json').read_text(encoding='utf-8')
        )
        require(quick_restart_metadata['components'] == [
            'optimizer', 'scheduler', 'scaler', 'rng'
        ], 'Quick checkpoint restart components are incomplete')
        require(quick_restart_metadata['global_step'] == quick_start_step + 1, 'Quick checkpoint metadata step mismatch')
        require(quick_manifest.global_step == quick_start_step + 1, 'Quick checkpoint manifest step mismatch')
        print('================================================================')
        print('QUICK TEST ONLY — FULL EPOCH NOT COMPLETED')
        print('Restartable quick-test checkpoint:', quick_test_checkpoint_dir)
        print('================================================================')
        break

    if not QUICK_TEST_MODE:
        production_optimizer_steps = math.ceil(
            TRAIN_ROWS
            / (per_device_batch_size * gradient_accumulation_steps)
        )
        while production_trainer.global_step < production_optimizer_steps:
            step_target = min(
                production_trainer.global_step + checkpoint_interval,
                production_optimizer_steps,
            )
            chunk_losses = production_trainer.train(max_steps=step_target)
            next_ordered_row = production_train_stream.next_ordered_row
            periodic_checkpoint_dir = os.path.join(
                run_root,
                'production-checkpoints',
                f'step-{production_trainer.global_step:08d}',
            )
            os.makedirs(os.path.dirname(periodic_checkpoint_dir), exist_ok=True)
            periodic_metrics = {
                'mode': 'production_in_progress',
                'latest_loss': chunk_losses[-1],
                'next_ordered_row': next_ordered_row,
            }
            periodic_manifest = production_trainer.save(
                periodic_checkpoint_dir,
                metrics=periodic_metrics,
            )
            complete_restartable_checkpoint(
                periodic_checkpoint_dir, production_trainer,
                production_train_stream, production_scaler, periodic_manifest,
            )
            print(
                'Drive checkpoint',
                production_trainer.global_step,
                '| next ordered row',
                next_ordered_row,
            )
        next_ordered_row = production_train_stream.next_ordered_row
        require(next_ordered_row == TRAIN_ROWS, 'Production did not consume exactly 50,000 ordered rows')
        require(production_trainer.global_step == production_optimizer_steps, 'Production optimizer step count mismatch')
        production_trainer.epoch = EPOCHS
        require(production_trainer.epoch == 1, 'Production epoch count mismatch')
        print('PRODUCTION: ordered 50,000-row epoch complete')


In [ ]:
if QUICK_TEST_MODE:
    print('Validation skipped: QUICK TEST ONLY — FULL EPOCH NOT COMPLETED')
else:
    from babel_training.validation import validate_embeddings
    fixed_validation_stream = OrderedInterviewStream(
        'validation', VALIDATION_ROWS
    )
    fixed_validation_rows = list(fixed_validation_stream)
    require(len(fixed_validation_rows) == VALIDATION_ROWS, 'Validation row count mismatch')
    validation_loader = DataLoader(
        fixed_validation_rows,
        batch_size=per_device_batch_size,
        collate_fn=collator,
        num_workers=0,
    )
    article_keys, student_chunks, teacher_chunks = [], [], []
    production_trainer.model.eval()
    device = next(production_trainer.model.parameters()).device
    with torch.no_grad():
        for batch in validation_loader:
            student_chunks.append(
                production_trainer.model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                ).float().cpu().numpy()
            )
            teacher_chunks.append(batch['teacher_vector'].float().numpy())
            article_keys.extend(batch['article_key'])
    student_vectors = np.concatenate(student_chunks)
    teacher_vectors = np.concatenate(teacher_chunks)
    student_norms = np.linalg.norm(student_vectors, axis=1)
    teacher_norms = np.linalg.norm(teacher_vectors, axis=1)
    invalid_student_vector_count = int(np.count_nonzero(
        ~np.all(np.isfinite(student_vectors), axis=1)
        | ~np.isfinite(student_norms)
        | (student_norms <= 0)
    ))
    invalid_teacher_vector_count = int(np.count_nonzero(
        ~np.all(np.isfinite(teacher_vectors), axis=1)
        | ~np.isfinite(teacher_norms)
        | (teacher_norms <= 0)
    ))
    report = validate_embeddings(
        article_keys,
        student_vectors,
        teacher_vectors,
        dataset_revision=dataset_revision,
        model_revision=config.model_revision,
        tokenizer_revision=config.model_revision,
        dataset_repo_id=DATASET_REPO_ID,
        dataset_config=DATASET_CONFIG,
        dataset_manifest_sha256=MANIFEST_SHA256,
        dataset_readiness_sha256=dataset_readiness_sha256,
        subset='fixed-interview-5000',
    )
    invalid_vector_count = report.invalid_vector_count
    validation_summary = {
        **report.to_dict(),
        'invalid_student_vector_count': invalid_student_vector_count,
        'invalid_teacher_vector_count': invalid_teacher_vector_count,
        'invalid_vector_count': invalid_vector_count,
    }
    validation_report_path = os.path.join(run_root, 'validation-report.json')
    Path(validation_report_path).write_text(
        json.dumps(validation_summary, sort_keys=True, separators=(',', ':')) + '\n',
        encoding='utf-8',
    )
    print('VALIDATION PASS: 5,000 fixed rows', validation_summary['metrics'])
    print('Invalid vectors:', {
        'student': invalid_student_vector_count,
        'teacher': invalid_teacher_vector_count,
        'joint': invalid_vector_count,
    })


In [ ]:
if QUICK_TEST_MODE:
    print('Production-final save skipped: quick-test checkpoint is already restartable.')
else:
    def checkpoint_tree_sha256(path):
        digest = hashlib.sha256()
        root = Path(path)
        for item in sorted(candidate for candidate in root.rglob('*') if candidate.is_file()):
            digest.update(item.relative_to(root).as_posix().encode('utf-8'))
            digest.update(b'\0')
            digest.update(hashlib.sha256(item.read_bytes()).digest())
        return digest.hexdigest()

    final_checkpoint_dir = os.path.join(run_root, 'production-final')
    final_checkpoint_fingerprint = production_trainer.validation_fingerprint()
    final_metrics = {
        **report.metrics,
        'invalid_student_vector_count': invalid_student_vector_count,
        'invalid_teacher_vector_count': invalid_teacher_vector_count,
        'invalid_vector_count': invalid_vector_count,
        'next_ordered_row': production_train_stream.next_ordered_row,
    }
    final_manifest = production_trainer.save(
        final_checkpoint_dir,
        metrics=final_metrics,
    )
    final_manifest = complete_restartable_checkpoint(
        final_checkpoint_dir, production_trainer, production_train_stream,
        production_scaler, final_manifest,
    )
    require(final_manifest.epoch == EPOCHS, 'Final checkpoint epoch mismatch')
    require(bool(final_manifest.loader_state), 'Final checkpoint loader state is missing')
    final_checkpoint_identity = checkpoint_tree_sha256(final_checkpoint_dir)
    print('Saved production-final restartable checkpoint:', final_checkpoint_identity)


In [ ]:
if QUICK_TEST_MODE:
    print('Final reload skipped in quick-test mode.')
else:
    fingerprint_before_reload = final_checkpoint_fingerprint
    production_trainer.reload(final_checkpoint_dir)
    fingerprint_after_reload = production_trainer.validation_fingerprint()
    require(fingerprint_after_reload == fingerprint_before_reload, 'Final checkpoint fingerprint mismatch')
    require(production_trainer.global_step == production_optimizer_steps, 'Reloaded global step mismatch')
    require(production_train_stream.next_ordered_row == TRAIN_ROWS, 'Reloaded ordered cursor mismatch')
    print('Final checkpoint reload/fingerprint PASS')


In [ ]:
if QUICK_TEST_MODE:
    print('Isolated resume verification is reserved for the production run.')
else:
    import shutil
    periodic_checkpoints = sorted(
        Path(run_root, 'production-checkpoints').glob('step-*')
    )
    resume_candidates = list(periodic_checkpoints)
    if RESUME_CHECKPOINT_DIR is not None:
        resume_candidates.append(Path(RESUME_CHECKPOINT_DIR))
    def candidate_global_step(path):
        return json.loads(
            Path(path, 'manifest.json').read_text(encoding='utf-8')
        )['global_step']

    resume_candidates.sort(key=candidate_global_step)
    resumable_source = next(
        path
        for path in reversed(resume_candidates)
        if candidate_global_step(path) < production_optimizer_steps
        and json.loads(Path(path, 'manifest.json').read_text())['loader_state']
    )
    resume_verification_dir = os.path.join(run_root, 'resume-verification')
    shutil.copytree(resumable_source, resume_verification_dir)
    final_identity_before_resume = checkpoint_tree_sha256(final_checkpoint_dir)
    copied_identity_before_resume = checkpoint_tree_sha256(resume_verification_dir)
    production_trainer.reload(resume_verification_dir)
    saved_step = production_trainer.global_step
    resume_losses = production_trainer.train(max_steps=saved_step + 1)
    require(production_trainer.global_step == saved_step + 1, 'Resume verification did not take one step')
    require(bool(resume_losses), 'Resume verification produced no loss')
    require(checkpoint_tree_sha256(resume_verification_dir) == copied_identity_before_resume, 'Resume verification copy was mutated')
    require(checkpoint_tree_sha256(final_checkpoint_dir) == final_identity_before_resume, 'Production-final checkpoint was mutated')
    production_trainer.reload(final_checkpoint_dir)
    require(production_trainer.validation_fingerprint() == final_checkpoint_fingerprint, 'Final fingerprint was not restored after resume verification')
    print('Isolated one-step resume PASS; production-final checkpoint remained immutable.')


In [ ]:
if QUICK_TEST_MODE:
    print('Artifact export skipped: quick-test weights are not production artifacts.')
else:
    from safetensors.torch import save_file

    def canonical_json_bytes(value):
        return (
            json.dumps(
                value,
                sort_keys=True,
                separators=(',', ':'),
                ensure_ascii=False,
                allow_nan=False,
            ) + '\n'
        ).encode('utf-8')

    final_student = production_trainer.accelerator.unwrap_model(
        production_trainer.model
    )
    projection_tensors, adapter_tensors, adapter_config = (
        final_student.export_components()
    )
    require(projection_tensors['weight'].shape == (100, 1024), 'Projection weight shape mismatch')
    require(projection_tensors['bias'].shape == (100,), 'Projection bias shape mismatch')
    require(bool(adapter_tensors), 'LoRA adapter export is empty')
    artifact_dir = Path(run_root, 'distilled-artifact')
    artifact_dir.mkdir()
    save_file(projection_tensors, artifact_dir / 'projection.safetensors')
    save_file(adapter_tensors, artifact_dir / 'adapter_model.safetensors')
    complete_training_config = {
        **training_config,
        'quick_test_mode': False,
        'completed_ordered_train_rows': TRAIN_ROWS,
        'completed_epochs': EPOCHS,
        'final_global_step': production_trainer.global_step,
    }
    export_documents = {
        'adapter_config.json': adapter_config,
        'training_config.json': complete_training_config,
        'validation_report.json': validation_summary,
        'final_checkpoint_identity.json': {
            'schema_version': 1,
            'tree_sha256': final_checkpoint_identity,
            'epoch': final_manifest.epoch,
            'global_step': final_manifest.global_step,
            'next_ordered_row': production_train_stream.next_ordered_row,
        },
    }
    for filename, document in export_documents.items():
        Path(artifact_dir, filename).write_bytes(canonical_json_bytes(document))
    artifact_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(artifact_dir.iterdir())
        if path.is_file()
    }
    artifact_identity = {
        'artifact_schema': 'babel-distillation-2016-interview-v1',
        'source': {'commit_sha': SOURCE_COMMIT_SHA},
        'model': {
            'id': config.model_id,
            'revision': MODEL_REVISION,
            'tokenizer_revision': MODEL_REVISION,
        },
        'dataset': {
            'repo_id': DATASET_REPO_ID,
            'config': 'distillation_2016_interview',
            'commit_sha': DATASET_REVISION,
            'manifest_sha256': MANIFEST_SHA256,
            'readiness_sha256': dataset_readiness_sha256,
            'counts': EXPECTED_COUNTS,
            'ordered_identity_sha256': EXPECTED_ORDERED_SHA256,
            'parquet_sha256': EXPECTED_PARQUET_SHA256,
            'test_usage': 'identity metadata only; examples unopened',
        },
        'protocol': {
            'smoke_rows': SMOKE_ROWS,
            'train_rows': TRAIN_ROWS,
            'validation_rows': VALIDATION_ROWS,
            'epochs': EPOCHS,
            'max_length': config.max_length,
        },
        'lora': adapter_config,
        'projection': {'input_dimension': 1024, 'output_dimension': 100},
        'training_config': complete_training_config,
        'validation': validation_summary,
        'final_checkpoint': export_documents['final_checkpoint_identity.json'],
        'artifact_hashes': artifact_hashes,
    }
    artifact_id = hashlib.sha256(canonical_json_bytes(artifact_identity)).hexdigest()
    artifact_manifest = {
        'artifact_id': artifact_id,
        'immutable': True,
        **artifact_identity,
        'publication': {
            'repo_id': DESTINATION_MODEL_REPO,
            'private': True,
            'artifact_payload_commit_sha': None,
        },
    }
    artifact_manifest_path = artifact_dir / 'artifact_manifest.json'
    artifact_manifest_path.write_bytes(canonical_json_bytes(artifact_manifest))
    print('Immutable LoRA + 100d projection export prepared:', artifact_id)
    print('Artifact hashes:', artifact_hashes)


In [ ]:
import re


def classify_publication_state(existing_artifact_paths, payload_path_set, manifest_repo_path):
    expected_complete_paths = payload_path_set | {manifest_repo_path}
    if not existing_artifact_paths:
        return 'new'
    if existing_artifact_paths == payload_path_set:
        return 'payload_only'
    if existing_artifact_paths == expected_complete_paths:
        return 'complete'
    raise RuntimeError('Remote artifact prefix is partial or conflicting')


def verify_remote_artifact_hashes(
    repo_id,
    revision,
    expected_hashes,
    expected_sizes,
    token,
    download_fn,
):
    remote_artifact_hashes = {}
    for repo_path in sorted(expected_hashes):
        downloaded = Path(download_fn(
            repo_id,
            filename=repo_path,
            repo_type='model',
            revision=revision,
            token=token,
        ))
        raw = downloaded.read_bytes()
        actual_sha256 = hashlib.sha256(raw).hexdigest()
        actual_size = len(raw)
        if (
            actual_sha256 != expected_hashes[repo_path]
            or actual_size != expected_sizes[repo_path]
        ):
            raise RuntimeError(f'Remote artifact byte mismatch: {repo_path}')
        remote_artifact_hashes[repo_path] = {
            'sha256': actual_sha256,
            'size': actual_size,
        }
    return remote_artifact_hashes


def publish_private_artifact(
    api,
    repo_id,
    artifact_id_value,
    artifact_directory,
    local_manifest,
    local_manifest_path,
    local_artifact_hashes,
    token,
    add_operation_factory,
    download_fn,
    canonical_json_fn,
):
    api.create_repo(
        repo_id=repo_id,
        repo_type='model',
        private=True,
        exist_ok=True,
        token=token,
    )
    repo_info = api.model_info(repo_id, revision='main', token=token)
    if getattr(repo_info, 'private', False) is not True:
        raise RuntimeError('Destination model repository is not private')
    artifact_prefix = f'artifacts/{artifact_id_value}/'
    payload_paths = sorted(
        path for path in artifact_directory.iterdir()
        if path.is_file() and path.name != 'artifact_manifest.json'
    )
    payload_repo_paths = {
        artifact_prefix + path.name: path for path in payload_paths
    }
    payload_path_set = set(payload_repo_paths)
    manifest_repo_path = artifact_prefix + 'artifact_manifest.json'
    expected_complete_paths = payload_path_set | {manifest_repo_path}
    existing_artifact_paths = {
        sibling.rfilename
        for sibling in getattr(repo_info, 'siblings', [])
        if sibling.rfilename.startswith(artifact_prefix)
    }
    publication_state = classify_publication_state(
        existing_artifact_paths, payload_path_set, manifest_repo_path
    )
    payload_hashes = {
        artifact_prefix + filename: sha256
        for filename, sha256 in local_artifact_hashes.items()
    }
    payload_sizes = {
        repo_path: path.stat().st_size
        for repo_path, path in payload_repo_paths.items()
    }

    if publication_state == 'complete':
        remote_manifest_path = Path(download_fn(
            repo_id,
            filename=manifest_repo_path,
            repo_type='model',
            revision=repo_info.sha,
            token=token,
        ))
        remote_manifest_bytes = remote_manifest_path.read_bytes()
        remote_manifest = json.loads(remote_manifest_bytes)
        publication = remote_manifest.get('publication')
        if not isinstance(publication, dict) or set(publication) != {
            'repo_id', 'private', 'artifact_payload_commit_sha'
        }:
            raise RuntimeError('Remote artifact manifest identity conflicts')
        artifact_payload_commit_sha = publication['artifact_payload_commit_sha']
        expected_publication = {
            'repo_id': repo_id,
            'private': True,
            'artifact_payload_commit_sha': artifact_payload_commit_sha,
        }
        expected_manifest = {
            **local_manifest,
            'publication': expected_publication,
        }
        if (
            re.fullmatch(r'[a-f0-9]{40}', str(artifact_payload_commit_sha)) is None
            or publication != expected_publication
            or remote_manifest_bytes != canonical_json_fn(expected_manifest)
        ):
            raise RuntimeError('Remote artifact manifest identity conflicts')
        verify_remote_artifact_hashes(
            repo_id,
            artifact_payload_commit_sha,
            payload_hashes,
            payload_sizes,
            token,
            download_fn,
        )
        artifact_manifest_sha256 = hashlib.sha256(
            remote_manifest_bytes
        ).hexdigest()
        final_hashes = {
            **payload_hashes,
            manifest_repo_path: artifact_manifest_sha256,
        }
        final_sizes = {
            **payload_sizes,
            manifest_repo_path: len(remote_manifest_bytes),
        }
        remote_artifact_hashes = verify_remote_artifact_hashes(
            repo_id,
            repo_info.sha,
            final_hashes,
            final_sizes,
            token,
            download_fn,
        )
        local_manifest.clear()
        local_manifest.update(remote_manifest)
        local_manifest_path.write_bytes(remote_manifest_bytes)
        hub_commit_sha = repo_info.sha
    else:
        if publication_state == 'new':
            payload_commit = api.create_commit(
                repo_id=repo_id,
                repo_type='model',
                revision='main',
                parent_commit=repo_info.sha,
                operations=[
                    add_operation_factory(
                        path_in_repo=repo_path,
                        path_or_fileobj=str(path),
                    )
                    for repo_path, path in sorted(payload_repo_paths.items())
                ],
                commit_message=f'Publish interview artifact payload {artifact_id_value}',
                token=token,
            )
            artifact_payload_commit_sha = payload_commit.oid
        else:
            artifact_payload_commit_sha = repo_info.sha
        if re.fullmatch(r'[a-f0-9]{40}', str(artifact_payload_commit_sha)) is None:
            raise RuntimeError('Artifact payload commit SHA is malformed')
        verify_remote_artifact_hashes(
            repo_id,
            artifact_payload_commit_sha,
            payload_hashes,
            payload_sizes,
            token,
            download_fn,
        )
        exact_publication = {
            'repo_id': repo_id,
            'private': True,
            'artifact_payload_commit_sha': artifact_payload_commit_sha,
        }
        local_manifest['publication'] = exact_publication
        local_manifest_path.write_bytes(canonical_json_fn(local_manifest))
        artifact_manifest_sha256 = hashlib.sha256(
            local_manifest_path.read_bytes()
        ).hexdigest()
        manifest_commit = api.create_commit(
            repo_id=repo_id,
            repo_type='model',
            revision='main',
            parent_commit=artifact_payload_commit_sha,
            operations=[
                add_operation_factory(
                    path_in_repo=manifest_repo_path,
                    path_or_fileobj=str(local_manifest_path),
                )
            ],
            commit_message=f'Finalize interview artifact manifest {artifact_id_value}',
            token=token,
        )
        hub_commit_sha = manifest_commit.oid
        if re.fullmatch(r'[a-f0-9]{40}', str(hub_commit_sha)) is None:
            raise RuntimeError('Final manifest commit SHA is malformed')
        final_hashes = {
            **payload_hashes,
            manifest_repo_path: artifact_manifest_sha256,
        }
        final_sizes = {
            **payload_sizes,
            manifest_repo_path: local_manifest_path.stat().st_size,
        }
        remote_artifact_hashes = verify_remote_artifact_hashes(
            repo_id,
            hub_commit_sha,
            final_hashes,
            final_sizes,
            token,
            download_fn,
        )

    verified_info = api.model_info(repo_id, revision=hub_commit_sha, token=token)
    if getattr(verified_info, 'private', False) is not True:
        raise RuntimeError('Published model repository is not private')
    if verified_info.sha != hub_commit_sha:
        raise RuntimeError('Published Hub commit identity mismatch')
    if set(remote_artifact_hashes) != expected_complete_paths:
        raise RuntimeError('Remote artifact verification is incomplete')
    return {
        'publication_state': publication_state,
        'artifact_payload_commit_sha': artifact_payload_commit_sha,
        'hub_commit_sha': hub_commit_sha,
        'artifact_manifest_sha256': artifact_manifest_sha256,
        'remote_artifact_hashes': remote_artifact_hashes,
    }


if QUICK_TEST_MODE:
    print('Hub publication skipped: quick-test artifacts are never published.')
else:
    from huggingface_hub import CommitOperationAdd

    publication_evidence = publish_private_artifact(
        HfApi(),
        DESTINATION_MODEL_REPO,
        artifact_id,
        artifact_dir,
        artifact_manifest,
        artifact_manifest_path,
        artifact_hashes,
        HF_TOKEN,
        CommitOperationAdd,
        hf_hub_download,
        canonical_json_bytes,
    )
    artifact_payload_commit_sha = publication_evidence[
        'artifact_payload_commit_sha'
    ]
    hub_commit_sha = publication_evidence['hub_commit_sha']
    artifact_manifest_sha256 = publication_evidence[
        'artifact_manifest_sha256'
    ]
    remote_artifact_hashes = publication_evidence[
        'remote_artifact_hashes'
    ]
    print('PRIVATE HUB PUBLICATION PASS')
    print('Repository:', DESTINATION_MODEL_REPO)
    print('Recovery state:', publication_evidence['publication_state'])
    print('Artifact payload commit SHA:', artifact_payload_commit_sha)
    print('Final manifest commit SHA:', hub_commit_sha)
    print('Final artifact manifest SHA-256:', artifact_manifest_sha256)
    print('Verified remote artifact bytes:', remote_artifact_hashes)
